In [ ]:
import ctypes
import _ctypes
import os
from IPython.display import display
import ipywidgets as widgets
from pynq import PL, Overlay, allocate, MMIO
from pynq.lib.video import VideoMode
import cv2
import numpy as np
from enum import Enum
import time

projectDir = "/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator"
os.chdir(projectDir)
print("Working directory set to:", os.getcwd())

PL.reset()
overlay = Overlay(projectDir + "/VivadoHW.bit")
overlay.download()

try:
  _ctypes.dlclose(lib_pipeline._handle)
except NameError:
  pass

lib_pipeline = ctypes.CDLL("./liborb_pipeline.so")

class PipelineResult(ctypes.Structure):
  _fields_ = [
      ("good_matches", ctypes.c_int),
      ("total_corners", ctypes.c_int),
      ("fps", ctypes.c_double),
      ("jpeg_size", ctypes.c_int),
      ("unpack_ms", ctypes.c_double),
      ("match_ms", ctypes.c_double),
      ("ransac_ms", ctypes.c_double),
      ("draw_encode_ms", ctypes.c_double),
      ("cpp_total_ms", ctypes.c_double),
  ]

lib_pipeline.init_ref_descriptors.argtypes = [ctypes.c_void_p, ctypes.c_uint32]
lib_pipeline.init_ref_descriptors.restype = ctypes.c_int

lib_pipeline.process_and_draw_frame.argtypes = [
    ctypes.c_void_p,
    ctypes.c_uint32,
    ctypes.POINTER(ctypes.c_uint8),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_double,
    ctypes.POINTER(ctypes.c_uint8),
    ctypes.c_int,
    ctypes.POINTER(PipelineResult),
]
lib_pipeline.process_and_draw_frame.restype = ctypes.c_int

Working directory set to: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator


In [ ]:
class VideoSource(Enum):
    HDMI = (0, 0)
    TPG_COLOR_BAR_BOX = (1, 1)
    TPG_GRID_BOX = (2, 1)
    PS_TO_PL = (3, 2)

IN_WIDTH, IN_HEIGHT = 1280, 720
IMAGES_DIR = projectDir + "/images"
VALID_IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')
available_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS)] if os.path.exists(IMAGES_DIR) else []
ref_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS) and '_ref' in f.lower()] if os.path.exists(IMAGES_DIR) else []

tpg = overlay.v_tpg_0
vdma = overlay.dma_block.axi_vdma_0
hdmi_vdma = overlay.hdmi.axi_vdma_0
dma = overlay.dma_block.axi_dma_0
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
fast_threshold_gpio = overlay.axi_gpios.axi_gpio_2
image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='100%'))
stream_switch = overlay.orb_pipeline.axis_switch_0
timing_mon = getattr(overlay.orb_pipeline, "orb_timing_monitor_0", None) or MMIO(0x43C20000, 0x10000)

def print_perf_timings(res=None, py_overhead_ms=0.0, active_frame_ms=0.0, total_loop_ms=0.0):
    total_cyc = timing_mon.read(0x00)
    t_gray = timing_mon.read(0x04)
    t_gauss = timing_mon.read(0x08)
    t_fast = timing_mon.read(0x0C)
    t_brief = timing_mon.read(0x10)
    hw_frames = timing_mon.read(0x14)

    d_gray = t_gray
    d_gauss = max(0, t_gauss - t_gray)
    d_fast = max(0, t_fast - t_gauss)
    d_brief = max(0, t_brief - t_fast)
    d_drain = max(0, total_cyc - t_brief)

    print(
        f"[PL HW #{hw_frames}] Total: {total_cyc * 1e-5:.3f} ms ({total_cyc} cyc) | "
        f"Latency -> Gray: {d_gray * 0.01:.2f} us, Gauss: {d_gauss * 0.01:.2f} us, "
        f"FAST: {d_fast * 0.01:.2f} us, rBRIEF: {d_brief * 0.01:.2f} us | "
        f"Drain+Merge: {d_drain * 1e-5:.3f} ms"
    )
    if res is not None:
        eff_fps = (1000.0 / total_loop_ms) if total_loop_ms > 0 else 0.0
        print(
            f"[PS C++] Total: {res.cpp_total_ms:.2f} ms | "
            f"Unpack: {res.unpack_ms:.2f} ms, Match: {res.match_ms:.2f} ms, "
            f"RANSAC: {res.ransac_ms:.2f} ms, Draw+JPEG: {res.draw_encode_ms:.2f} ms\n"
            f"[TOTAL FRAME] Python overhead: {py_overhead_ms:.2f} ms | "
            f"Active PS+Py: {active_frame_ms:.2f} ms | "
            f"Full Frame Period: {total_loop_ms:.2f} ms ({eff_fps:.1f} FPS)"
        )

current_ps_frame = None
ps_to_pl_selected_image = 0
selected_source = None
current_desc_count = 0
frame_count = 0
RANSAC_THRESHOLD = 4.0
FAST_THRESHOLD = 90
HAMMING_THRESHOLD = 60

In [6]:
fast_threshold_gpio.channel1.write(FAST_THRESHOLD, 0xFFFFFFFF)
dummy_desc_buf = allocate(shape=(65536,), dtype=np.uint32)
current_transfer_bytes = 0


def process_and_rearm_dma():
  global current_desc_count, current_transfer_bytes
  status = dma.read(0x34)
  if (status & 0x1) or (status & 0x10):
    dma.write(0x30, 0x4)
    while dma.read(0x30) & 0x4:
      pass
    dma.recvchannel.start()
    dma.recvchannel.transfer(dummy_desc_buf)
    return
  if status & 0x2:
    current_transfer_bytes = dma.read(0x58)
    nbrOfDescriptors = current_transfer_bytes // 64
    current_desc_count = max(0, nbrOfDescriptors - 1) if nbrOfDescriptors > 0 else 0
    dma.recvchannel.transfer(dummy_desc_buf)


def setupVideoSource(new_src: VideoSource, force_reload=False):
  global selected_source, ps_to_pl_selected_image, current_ps_frame
  if selected_source == new_src and new_src != VideoSource.PS_TO_PL and not force_reload:
    return

  tpg.write(0x0000, 0x0)
  stream_switch.write(0x040, 0x80000000)
  stream_switch.write(0x000, 0x2)

  if vdma.readchannel.running:
    vdma.readchannel.stop()
  if hdmi_vdma.writechannel.running:
    hdmi_vdma.writechannel.stop()
  if hdmi_vdma.readchannel.running:
    hdmi_vdma.readchannel.stop()

  selected_source = new_src
  imageResolution.channel1.write(IN_HEIGHT, 0xFFFFFFFF)
  imageResolution.channel2.write(IN_WIDTH, 0xFFFFFFFF)
  tpg.write(0x0010, IN_HEIGHT)
  tpg.write(0x0018, IN_WIDTH)

  if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
    tpg.write(0x0020, 9)
    tpg.write(0x0028, 1)
    tpg.write(0x0038, 8)
    tpg.write(0x00D8, 0)
    tpg.write(0x0058, 150)
    tpg.write(0x0078, 200)
    tpg.write(0x0080, 200)
    tpg.write(0x0088, 50)
    tpg.write(0x0090, 150)
  elif selected_source == VideoSource.TPG_GRID_BOX:
    tpg.write(0x0020, 11)
    tpg.write(0x0028, 1)
    tpg.write(0x0038, 8)
    tpg.write(0x00D8, 0)
    tpg.write(0x0058, 150)
    tpg.write(0x0078, 200)
    tpg.write(0x0080, 200)
    tpg.write(0x0088, 50)
    tpg.write(0x0090, 150)

  if not dma.recvchannel.running:
    dma.recvchannel.start()
  if dma.recvchannel.idle:
    dma.recvchannel.transfer(dummy_desc_buf)

  if not vdma.writechannel.running:
    vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.writechannel.start()

  vdma.readchannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
  vdma.readchannel.start()
  vdma.readchannel.irqframecount = 1

  stream_switch.write(0x040, selected_source.value[1])
  stream_switch.write(0x000, 0x2)

  if selected_source == VideoSource.HDMI:
    hdmi_mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    hdmi_vdma.readchannel.mode = hdmi_mode
    hdmi_vdma.writechannel.mode = hdmi_mode
    hdmi_vdma.readchannel.start()
    hdmi_vdma.writechannel.start()
    hdmi_vdma.readchannel.tie(hdmi_vdma.writechannel)
  elif selected_source in [VideoSource.TPG_COLOR_BAR_BOX, VideoSource.TPG_GRID_BOX]:
    tpg.write(0x0000, 0x81)
  elif selected_source == VideoSource.PS_TO_PL:
    if available_images:
      current_image_path = available_images[ps_to_pl_selected_image]
      print(
          f"Loading image [{ps_to_pl_selected_image + 1}/{len(available_images)}]"
          f" for PS_TO_PL: {current_image_path}"
      )
      img = cv2.imread(current_image_path)
      h, w = img.shape[:2]

      scale = min(IN_WIDTH / w, IN_HEIGHT / h)
      nw, nh = int(w * scale), int(h * scale)
      resized_img = cv2.resize(img, (nw, nh))

      current_ps_frame = np.full((IN_HEIGHT, IN_WIDTH, 3), 255, dtype=np.uint8)
      top = (IN_HEIGHT - nh) // 2
      left = (IN_WIDTH - nw) // 2
      current_ps_frame[top : top + nh, left : left + nw] = resized_img

      buf = vdma.writechannel.newframe()
      np.copyto(buf, current_ps_frame)
      vdma.writechannel.writeframe(buf)

      ps_to_pl_selected_image = (ps_to_pl_selected_image + 1) % len(available_images)


if not dma.recvchannel.running:
  dma.recvchannel.start()
dma.recvchannel.transfer(dummy_desc_buf)

print("Capturing Baseline Reference Descriptors...")
ref_image_path = ref_images[0]
ps_to_pl_selected_image = available_images.index(ref_image_path)
setupVideoSource(VideoSource.PS_TO_PL, force_reload=True)

timeout_counter = 0
stable_frames = 0
last_corner_count = -1

while stable_frames < 5 and timeout_counter < 60:
  while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
    process_and_rearm_dma()
    time.sleep(0.001)

  _ = vdma.readchannel.readframe()
  process_and_rearm_dma()
  timeout_counter += 1

  if current_desc_count > 0:
    if current_desc_count == last_corner_count:
      stable_frames += 1
    else:
      stable_frames = 1
    last_corner_count = current_desc_count
  else:
    stable_frames = 0
    last_corner_count = 0

locked_count = lib_pipeline.init_ref_descriptors(
    dummy_desc_buf.ctypes.data, current_transfer_bytes
)
print(f"SUCCESS: Locked {locked_count} reference descriptors in C++.")
print_perf_timings()

image_widget.layout.width = "100%"
display(image_widget)

old_button_value = buttons.read(0)
MAX_JPEG_SIZE = 1024 * 1024
jpeg_out_buf = (ctypes.c_uint8 * MAX_JPEG_SIZE)()
pipeline_res = PipelineResult()
prev_frame_done = time.perf_counter()


try:
  while True:
    process_and_rearm_dma()
    button_pressed = False

    while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
      process_and_rearm_dma()
      
      button_value = buttons.read(0)
      if button_value != old_button_value and button_value > 0:
        button_pressed = True
        break

      r_status = vdma.readchannel._mmio.read(0x34)
      if r_status & 0x1:
        vdma.readchannel.stop()
        vdma.readchannel.start()
        vdma.readchannel.irqframecount = 1
      if r_status & 0x8980:
        vdma.readchannel._mmio.write(0x34, 0x8980)
      time.sleep(0.001)

    if button_pressed:
      if button_value & 0x1:
        setupVideoSource(VideoSource.HDMI)
      elif button_value & 0x2:
        setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
      elif button_value & 0x4:
        setupVideoSource(VideoSource.TPG_GRID_BOX)
      elif button_value & 0x8:
        setupVideoSource(VideoSource.PS_TO_PL, force_reload=True)
      old_button_value = button_value
      prev_frame_done = time.perf_counter()
      continue

    t_active_start = time.perf_counter()
    frame = vdma.readchannel.readframe()
    current_transfer_bytes = dma.read(0x58)
    process_and_rearm_dma()

    safe_frame = np.copy(frame)

    lib_pipeline.process_and_draw_frame(
        dummy_desc_buf.ctypes.data,
        current_transfer_bytes,
        safe_frame.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8)),
        IN_WIDTH,
        IN_HEIGHT,
        HAMMING_THRESHOLD,
        RANSAC_THRESHOLD,
        jpeg_out_buf,
        MAX_JPEG_SIZE,
        ctypes.byref(pipeline_res),
    )

    if pipeline_res.jpeg_size > 0:
      image_widget.value = bytes(jpeg_out_buf[: pipeline_res.jpeg_size])

    t_active_end = time.perf_counter()
    active_frame_ms = (t_active_end - t_active_start) * 1000.0
    py_overhead_ms = max(0.0, active_frame_ms - pipeline_res.cpp_total_ms)
    total_loop_ms = (t_active_end - prev_frame_done) * 1000.0
    prev_frame_done = t_active_end

    button_value = buttons.read(0)
    if button_value != old_button_value and button_value > 0:
      if button_value & 0x1:
        setupVideoSource(VideoSource.HDMI)
      elif button_value & 0x2:
        setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
      elif button_value & 0x4:
        setupVideoSource(VideoSource.TPG_GRID_BOX)
      elif button_value & 0x8:
        setupVideoSource(VideoSource.PS_TO_PL, force_reload=True)
    old_button_value = button_value
    
    frame_count += 1
    if frame_count % 30 == 0 or selected_source == VideoSource.PS_TO_PL:
      print_perf_timings(pipeline_res, py_overhead_ms, active_frame_ms, total_loop_ms)

except KeyboardInterrupt:
  print("\nStopping stream...")
finally:
  tpg.write(0x00, 0x0)
  if hdmi_vdma.writechannel.running:
    hdmi_vdma.writechannel.stop()
  if hdmi_vdma.readchannel.running:
    hdmi_vdma.readchannel.stop()
  if vdma.readchannel.running:
    vdma.readchannel.stop()
  if vdma.writechannel.running:
    vdma.writechannel.stop()
  dummy_desc_buf.close()
  print("Stream stopped cleanly.")

Capturing Baseline Reference Descriptors...
Loading image [3/9] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/coke_ref.jpg
SUCCESS: Locked 119 reference descriptors in C++.
[HW Frame #9] Total: 1350464 cyc (13.505 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), FAST: 7835 cyc (78.35 us), rBRIEF: 128588 cyc (1285.88 us) | Frame drain+merge: 1207547 cyc (12.075 ms)


Image(value=b'', format='jpeg', layout="Layout(width='100%')")

[HW Frame #12] Total: 1028738 cyc (10.287 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), FAST: 7835 cyc (78.35 us), rBRIEF: 128588 cyc (1285.88 us) | Frame drain+merge: 885821 cyc (8.858 ms)
[HW Frame #14] Total: 1736679 cyc (17.367 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), FAST: 7835 cyc (78.35 us), rBRIEF: 128588 cyc (1285.88 us) | Frame drain+merge: 1593762 cyc (15.938 ms)
[HW Frame #16] Total: 958834 cyc (9.588 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), FAST: 7835 cyc (78.35 us), rBRIEF: 128588 cyc (1285.88 us) | Frame drain+merge: 815917 cyc (8.159 ms)
[HW Frame #18] Total: 958834 cyc (9.588 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), FAST: 7835 cyc (78.35 us), rBRIEF: 128588 cyc (1285.88 us) | Frame drain+merge: 815917 cyc (8.159 ms)
[HW Frame #20] Total: 958834 cyc (9.588 ms) | First-px latency -> Gray: 1 cyc (0.01 us), Gauss: 6493 cyc (64.93 us), F